# Tool Use in Agents

This notebook adds a tutorial on tool calling without changing the existing `src/tools.py` path. It introduces a separate educational layer for tool registries, tool selection, and structured tool calls.

## Learning goals

- Understand why agents use tools.
- Learn the role of a tool registry.
- Practice tool selection and structured tool calls.
- Compare outputs from calculator, data, and search tools.


## Concept explanation

We start with the same environment check used throughout the course. This is especially helpful when you are switching between multiple Jupyter kernels or remote environments.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

This setup cell imports the additive tooling helpers from `src/tools_extended.py`. The original `src/tools.py` remains untouched, and the notebook focuses on a tutorial-friendly registry API.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.tools_extended import ToolCall, build_default_registry

pd.set_option('display.max_colwidth', 140)
registry = build_default_registry()


Agents use tools when retrieval and plain reasoning are not enough. A tool registry gives the agent a controlled list of what it is allowed to call. Structured tool calls make those calls explicit instead of hiding them inside free-form text.

Examples in this notebook:

- `calculator` for arithmetic
- `data_tool` for small table operations
- `search_tool` for simple similarity-style document search

## Implementation


In [ ]:
registry.list_tools()


Structured calls matter because they separate intent from execution. The cell below creates explicit tool call objects and runs them through the registry.


In [ ]:
structured_calls = [
    ToolCall('calculator', {'expression': '12 * 3 + 4'}),
    ToolCall('data_tool', {'rows': [{'team': 'A', 'score': 8}, {'team': 'B', 'score': 10}, {'team': 'A', 'score': 6}], 'column': 'score', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'launch date', 'documents': ['The launch date is May 5, 2025.', 'The pilot begins in March.', 'Finance approved the budget.']}),
]
structured_results = [registry.call(tool_call).to_dict() for tool_call in structured_calls]
pd.DataFrame(structured_results)


Tool selection is a separate problem from tool execution. A simple agent first decides which tools are relevant, then builds structured calls for only those tools.


In [ ]:
selection_examples = [
    'Calculate the pilot duration in days.',
    'Find which document mentions the launch date.',
    'Count how many rows are in this dataset.',
]
pd.DataFrame(
    {
        'query': selection_examples,
        'selected_tools': [', '.join(registry.select_tools(query)) for query in selection_examples],
    }
)


## Experiment

A useful experiment is to send several different structured requests through the registry and compare how stable the outputs are. This helps you see why typed tool inputs are easier to debug than free-form prompting alone.


In [ ]:
experiment_calls = [
    ToolCall('calculator', {'expression': '25 - 7'}),
    ToolCall('data_tool', {'rows': [{'latency': 0.8}, {'latency': 1.2}, {'latency': 1.0}], 'column': 'latency', 'operation': 'mean'}),
    ToolCall('search_tool', {'query': 'pilot window', 'documents': ['Pilot window: March 10, 2025 to April 4, 2025.', 'The governance memo explains ownership.', 'The FAQ lists tool guidance.']}),
]
experiment_results = [registry.call(tool_call).to_dict() for tool_call in experiment_calls]
pd.DataFrame(experiment_results)


## Result analysis

The outputs show three complementary tool patterns: arithmetic tools produce exact values, data tools summarize structured rows, and search tools return ranked candidate text. In a real agent, the selection policy determines which of these tools should be used for a given question.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'tool': 'calculator', 'best_for': 'precise arithmetic and simple derived values'},
        {'tool': 'data_tool', 'best_for': 'small structured datasets and aggregates'},
        {'tool': 'search_tool', 'best_for': 'finding relevant text before synthesis'},
    ]
)
analysis_frame


## Takeaways

- Tool registries keep agent capabilities explicit and safe.
- Tool selection is a reasoning problem; execution is an interface problem.
- Structured tool calls are easier to test and debug than raw text tool requests.
- This notebook adds new educational tooling without disturbing the existing tool path.
